In [1]:
import config

In [2]:
import os
print("Current working directory:", os.getcwd())
print("Files in directory:", os.listdir())
print("config.py exists?", os.path.exists("config.py"))

Current working directory: c:\Users\DELL\Documents\Projects\Notebooks\Capstone(Data Pioneers) files\Capstone_Model_Deployment
Files in directory: ['.git', '.gitignore', '.ipynb_checkpoints', 'anaconda_projects', 'Capstone_main.ipynb', 'config.py', 'data_loader.py', 'feature_engineering.py', 'inference.py', 'models', 'model_evaluation.py', 'model_training.py', 'plots', 'README.md', 'results', 'streamlit_app.py', '__pycache__']
config.py exists? True


In [3]:
import config
print("Config imported successfully!")
print("DATA_PATH:", config.DATA_PATH)

Config imported successfully!
DATA_PATH: C:\Users\DELL\Documents\Projects\Notebooks\Capstone(Data Pioneers) files\predictive_maintenance.csv


In [4]:
import sys
import os
sys.path.insert(0, os.getcwd())

# Import config first
import config

# Now import data loader
from data_loader import load_and_validate_data, split_data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             roc_curve, precision_recall_curve, f1_score, accuracy_score)
from imblearn.over_sampling import SMOTE
import pickle

print("✅ All imports successful")
print(f"Dataset path: {config.DATA_PATH}")
print(f"Models will be saved to: {config.MODELS_DIR}")

✅ All imports successful
Dataset path: C:\Users\DELL\Documents\Projects\Notebooks\Capstone(Data Pioneers) files\predictive_maintenance.csv
Models will be saved to: c:\Users\DELL\Documents\Projects\Notebooks\Capstone(Data Pioneers) files\Capstone_Model_Deployment\models


In [5]:
import config
from data_loader import load_and_validate_data, split_data

print("\n" + "="*80)
print("STEP 1: LOAD AND VALIDATE DATA")
print("="*80)

# Load the raw data
df = load_and_validate_data(config.DATA_PATH)
print(f"\n✓ Dataset loaded successfully: {df.shape}")

# Display first few rows
print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nBasic Statistics:")
print(df.describe())


STEP 1: LOAD AND VALIDATE DATA
[INFO] Loading dataset from: C:\Users\DELL\Documents\Projects\Notebooks\Capstone(Data Pioneers) files\predictive_maintenance.csv...
[INFO] Initial dataset shape: 10000 rows, 10 columns
[INFO] No missing values detected.
[INFO] No duplicate rows detected.
[INFO] Dropping non-predictive columns: ['UDI', 'Product ID', 'Failure Type']
[INFO] Target Class Distribution:
   Class 0: 9661 occurrences (96.61%)
   Class 1: 339 occurrences (3.39%)

✓ Dataset loaded successfully: (10000, 7)

First 5 rows:
  machine_type  air_temp  proc_temp  rot_speed  torque  tool_wear  failure
0            M     298.1      308.6       1551    42.8          0        0
1            L     298.2      308.7       1408    46.3          3        0
2            L     298.1      308.5       1498    49.4          5        0
3            L     298.2      308.6       1433    39.5          7        0
4            L     298.2      308.7       1408    40.0          9        0

Dataset Info:
<cla

In [6]:
# ============================================================================
# CELL 3: SPLIT DATA INTO TRAIN/VAL/TEST
# ============================================================================

print("\n" + "="*80)
print("STEP 2: STRATIFIED TRAIN/VALIDATION/TEST SPLIT (70/15/15)")
print("="*80)

# Split data
train_df, val_df, test_df = split_data(df)

# Extract features and target
X_train = train_df.drop(columns=[config.CLEAN_TARGET])
y_train = train_df[config.CLEAN_TARGET]

X_val = val_df.drop(columns=[config.CLEAN_TARGET])
y_val = val_df[config.CLEAN_TARGET]

X_test = test_df.drop(columns=[config.CLEAN_TARGET])
y_test = test_df[config.CLEAN_TARGET]

print(f"\nTrain set: {X_train.shape}")
print(f"Val set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

print(f"\n✅ Data split complete. Ready for feature engineering.")


STEP 2: STRATIFIED TRAIN/VALIDATION/TEST SPLIT (70/15/15)

[INFO] Data Splitting Summary:
   Train Set : 7000 rows (70.0%) | Failures: 237 (3.39%)
   Val Set   : 1500 rows (15.0%) | Failures:  51 (3.40%)
   Test Set  : 1500 rows (15.0%) | Failures:  51 (3.40%)

Train set: (7000, 6)
Val set: (1500, 6)
Test set: (1500, 6)

✅ Data split complete. Ready for feature engineering.


In [7]:
# ============================================================================
# COMPLETE TRAINING PIPELINE WITH ENGINEERED FEATURES
# ============================================================================
# This cell replaces the old cells 4, 5, 6, 7, and 8.
# It uses FeaturePipeline (from feature_engineering.py) to add engineered
# features, scale, encode, and then trains all 5 candidate models.
# All models are saved as {model_name}_final_model.pkl for Streamlit deployment.
# ============================================================================

print("\n" + "="*80)
print("STEP 3: FEATURE ENGINEERING & PREPROCESSING (using FeaturePipeline)")
print("="*80)

# Import necessary modules
from feature_engineering import FeaturePipeline, balance_training_data
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve
import xgboost as xgb
import pickle, json, os
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------------
# 1. Initialize and fit the FeaturePipeline on training data
# ----------------------------------------------------------------------------
print("\n[1] Initializing FeaturePipeline and fitting on training data...")
pipeline = FeaturePipeline()
pipeline.fit(train_df)   # This adds temp_diff, power_proxy, wear_per_speed, scales, and one-hot encodes
pipeline.save(config.MODELS_DIR)   # This saves scaler.pkl, encoder.pkl, pipeline_meta.pkl

# Transform all splits
X_train = pipeline.transform(train_df)
X_val = pipeline.transform(val_df)
X_test = pipeline.transform(test_df)

# Extract targets
y_train = train_df[config.CLEAN_TARGET].copy()
y_val = val_df[config.CLEAN_TARGET].copy()
y_test = test_df[config.CLEAN_TARGET].copy()

print(f"   Training features shape: {X_train.shape}")
print(f"   Validation features shape: {X_val.shape}")
print(f"   Test features shape: {X_test.shape}")

# ----------------------------------------------------------------------------
# 2. Balance the training data with SMOTE (only on training set)
# ----------------------------------------------------------------------------
print("\n[2] Applying SMOTE to balance training data...")
X_train_bal, y_train_bal = balance_training_data(X_train, y_train)
print(f"   Balanced training set size: {X_train_bal.shape[0]} rows")

# ----------------------------------------------------------------------------
# 3. Define the candidate models with their hyperparameters
#    (using the tuned values you found earlier)
# ----------------------------------------------------------------------------
print("\n[3] Defining candidate models...")
models = {
    'logistic_regression': LogisticRegression(
        max_iter=1000, 
        random_state=config.RANDOM_STATE, 
        class_weight='balanced'
    ),
    'decision_tree': DecisionTreeClassifier(
        max_depth=10, 
        random_state=config.RANDOM_STATE, 
        class_weight='balanced'
    ),
    'random_forest': RandomForestClassifier(
        n_estimators=100, 
        max_depth=15, 
        random_state=config.RANDOM_STATE, 
        class_weight='balanced', 
        n_jobs=-1
    ),
    'hist_gradient_boosting': HistGradientBoostingClassifier(
        learning_rate=0.1, 
        max_iter=200, 
        max_leaf_nodes=63, 
        min_samples_leaf=10, 
        random_state=config.RANDOM_STATE
    ),
    'xgboost': xgb.XGBClassifier(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.1,
        subsample=0.7,
        random_state=config.RANDOM_STATE,
        scale_pos_weight=len(y_train_bal[y_train_bal==0]) / len(y_train_bal[y_train_bal==1]),
        use_label_encoder=False,
        eval_metric='logloss'
    )
}

# ----------------------------------------------------------------------------
# 4. Train each model and evaluate on validation set
# ----------------------------------------------------------------------------
print("\n" + "="*80)
print("STEP 4: TRAINING & EVALUATING ALL CANDIDATE MODELS")
print("="*80)

results = {}
for name, model in models.items():
    print(f"\n[{list(models.keys()).index(name)+1}/{len(models)}] Training {name}...")
    model.fit(X_train_bal, y_train_bal)
    y_val_pred = model.predict(X_val)
    y_val_prob = model.predict_proba(X_val)[:, 1]
    
    results[name] = {
        'accuracy': accuracy_score(y_val, y_val_pred),
        'precision': precision_score(y_val, y_val_pred),
        'recall': recall_score(y_val, y_val_pred),
        'f1': f1_score(y_val, y_val_pred),
        'roc_auc': roc_auc_score(y_val, y_val_prob)
    }
    print(f"   ✓ F1: {results[name]['f1']:.4f}, ROC-AUC: {results[name]['roc_auc']:.4f}")

# Display comparison table
results_df = pd.DataFrame(results).T.sort_values('f1', ascending=False)
print("\n" + "="*80)
print("MODEL COMPARISON ON VALIDATION SET (RANKED BY F1-SCORE)")
print("="*80)
print(results_df.round(4))

# Identify the best model (highest F1)
best_model_name = results_df.index[0]
best_model = models[best_model_name]
print(f"\n🏆 Best model selected: {best_model_name} with F1 = {results_df.loc[best_model_name, 'f1']:.4f}")

# ----------------------------------------------------------------------------
# 5. Optimal threshold selection (using validation set)
# ----------------------------------------------------------------------------
print("\n" + "="*80)
print("STEP 5: OPTIMAL THRESHOLD OPTIMIZATION")
print("="*80)

y_val_prob = best_model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
print(f"   Optimal threshold (maximizing F1): {optimal_threshold:.4f}")

# ----------------------------------------------------------------------------
# 6. Final evaluation on test set with optimal threshold
# ----------------------------------------------------------------------------
print("\n" + "="*80)
print("STEP 6: FINAL TEST SET EVALUATION (with optimal threshold)")
print("="*80)

y_test_prob = best_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)

test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred),
    'recall': recall_score(y_test, y_test_pred),
    'f1': f1_score(y_test, y_test_pred),
    'roc_auc': roc_auc_score(y_test, y_test_prob)
}
print("   Test Set Performance:")
for metric, value in test_metrics.items():
    print(f"      {metric}: {value:.4f}")

# Print confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_test_pred)
print(f"\n   Confusion Matrix (Test):\n   {cm}")

# ----------------------------------------------------------------------------
# 7. Save all model artifacts for deployment
# ----------------------------------------------------------------------------
print("\n" + "="*80)
print("STEP 7: SAVING MODEL ARTIFACTS FOR STREAMLIT DEPLOYMENT")
print("="*80)

# The FeaturePipeline already saved scaler.pkl, encoder.pkl, and pipeline_meta.pkl
# Now save each trained model as {name}_final_model.pkl
for name, model in models.items():
    model_path = os.path.join(config.MODELS_DIR, f"{name}_final_model.pkl")
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"   ✅ Saved {name} model to {model_path}")

# Save optimal threshold for the best model (inference.py will load this)
threshold_path = os.path.join(config.RESULTS_DIR, f"{best_model_name}_best_threshold.txt")
os.makedirs(config.RESULTS_DIR, exist_ok=True)
with open(threshold_path, 'w') as f:
    f.write(str(optimal_threshold))
print(f"   ✅ Saved optimal threshold ({optimal_threshold:.4f}) for {best_model_name}")

# (Optional) Save a metadata JSON for reference
metadata = {
    'feature_names': pipeline.feature_names,
    'best_model': best_model_name,
    'optimal_threshold': float(optimal_threshold),
    'test_metrics': test_metrics
}
metadata_path = os.path.join(config.MODELS_DIR, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"   ✅ Saved metadata to {metadata_path}")

print("\n" + "="*80)
print("✅ ALL ARTIFACTS SUCCESSFULLY SAVED – READY FOR STREAMLIT")
print("="*80)
print("\nFiles in models/ directory:")
for file in sorted(os.listdir(config.MODELS_DIR)):
    print(f"   • {file}")


STEP 3: FEATURE ENGINEERING & PREPROCESSING (using FeaturePipeline)

[1] Initializing FeaturePipeline and fitting on training data...
[INFO] Feature pipeline artifacts saved to: c:\Users\DELL\Documents\Projects\Notebooks\Capstone(Data Pioneers) files\Capstone_Model_Deployment\models
   Training features shape: (7000, 11)
   Validation features shape: (1500, 11)
   Test features shape: (1500, 11)

[2] Applying SMOTE to balance training data...
[INFO] Class balance before SMOTE: [6763  237]
[INFO] Class balance after SMOTE: [6763 6763]
   Balanced training set size: 13526 rows

[3] Defining candidate models...

STEP 4: TRAINING & EVALUATING ALL CANDIDATE MODELS

[1/5] Training logistic_regression...
   ✓ F1: 0.2847, ROC-AUC: 0.9098

[2/5] Training decision_tree...
   ✓ F1: 0.6331, ROC-AUC: 0.9214

[3/5] Training random_forest...
   ✓ F1: 0.6891, ROC-AUC: 0.9826

[4/5] Training hist_gradient_boosting...
   ✓ F1: 0.8350, ROC-AUC: 0.9840

[5/5] Training xgboost...
   ✓ F1: 0.7288, ROC-AUC:

In [8]:
try:
    import xgboost as xgb
    print("✅ XGBoost installed")
except ImportError:
    print("❌ XGBoost not installed - install with: pip install xgboost")

✅ XGBoost installed


In [9]:
try:
    import xgboost as xgb
    print("✅ XGBoost installed")
except ImportError:
    print("❌ XGBoost not installed - install with: pip install xgboost")

✅ XGBoost installed


In [10]:
import pickle
import os

# The exact metadata
metadata = {
    "feature_names": [
        "air_temp",
        "proc_temp",
        "rot_speed",
        "torque",
        "tool_wear",
        "temp_diff",
        "power_proxy",
        "wear_per_speed",
        "machine_type"
    ],
    "numeric_features": [
        "air_temp",
        "proc_temp",
        "rot_speed",
        "torque",
        "tool_wear"
    ],
    "engineered_features": [
        "temp_diff",
        "power_proxy",
        "wear_per_speed"
    ],
    "categorical_feature": "machine_type",
    "categorical_values": ["H", "L", "M"],
    "optimal_threshold": 0.9738472104072571,
    "model_type": "XGBoost",
    "test_f1_score": 0.6823529411764706,
    "test_roc_auc": 0.9773880566719442,
    "test_precision": 0.8529411764705882,
    "test_recall": 0.5686274509803921,
    "test_accuracy": 0.982
}

os.makedirs("models", exist_ok=True)
with open("models/pipeline_meta.pkl", "wb") as f:
    pickle.dump(metadata, f)
    
print("✅ Successfully updated models/pipeline_meta.pkl with engineered features!")

✅ Successfully updated models/pipeline_meta.pkl with engineered features!


In [11]:
import pickle
import os
import config

# Load the XGBoost model (or any other)
model_path = os.path.join(config.MODELS_DIR, 'xgboost_final_model.pkl')
with open(model_path, 'rb') as f:
    model = pickle.load(f)

print("Features the model was trained on:")
print(model.feature_names_in_)

Features the model was trained on:
['air_temp' 'proc_temp' 'rot_speed' 'torque' 'tool_wear' 'temp_diff'
 'power_proxy' 'wear_per_speed' 'machine_type_H' 'machine_type_L'
 'machine_type_M']


In [12]:
import pickle
with open('models/pipeline_meta.pkl', 'rb') as f:
    meta = pickle.load(f)
print("Meta content:", meta)

Meta content: {'feature_names': ['air_temp', 'proc_temp', 'rot_speed', 'torque', 'tool_wear', 'temp_diff', 'power_proxy', 'wear_per_speed', 'machine_type'], 'numeric_features': ['air_temp', 'proc_temp', 'rot_speed', 'torque', 'tool_wear'], 'engineered_features': ['temp_diff', 'power_proxy', 'wear_per_speed'], 'categorical_feature': 'machine_type', 'categorical_values': ['H', 'L', 'M'], 'optimal_threshold': 0.9738472104072571, 'model_type': 'XGBoost', 'test_f1_score': 0.6823529411764706, 'test_roc_auc': 0.9773880566719442, 'test_precision': 0.8529411764705882, 'test_recall': 0.5686274509803921, 'test_accuracy': 0.982}


In [13]:
import pickle
with open('models/pipeline_meta.pkl', 'rb') as f:
    meta = pickle.load(f)
print("Meta length:", len(meta))
print("Model feature count:", len(model.feature_names_in_))

Meta length: 12
Model feature count: 11


In [14]:
import pickle
import os
import config

# Load any trained model to get the exact feature names
model_path = os.path.join(config.MODELS_DIR, 'xgboost_final_model.pkl')
with open(model_path, 'rb') as f:
    model = pickle.load(f)

# The model's feature names are already in the correct order
correct_feature_names = model.feature_names_in_.tolist()

# Overwrite pipeline_meta.pkl with this list
with open(os.path.join(config.MODELS_DIR, 'pipeline_meta.pkl'), 'wb') as f:
    pickle.dump(correct_feature_names, f)

print("✅ Updated pipeline_meta.pkl with correct feature names.")
print("Feature names:", correct_feature_names)
print("Number of features:", len(correct_feature_names))

✅ Updated pipeline_meta.pkl with correct feature names.
Feature names: ['air_temp', 'proc_temp', 'rot_speed', 'torque', 'tool_wear', 'temp_diff', 'power_proxy', 'wear_per_speed', 'machine_type_H', 'machine_type_L', 'machine_type_M']
Number of features: 11


In [15]:
import pickle
with open('models/pipeline_meta.pkl', 'rb') as f:
    data = pickle.load(f)
print(type(data))        # should be <class 'list'>
print(len(data))         # should be 11
print(data)              # should show all 11 feature names

<class 'list'>
11
['air_temp', 'proc_temp', 'rot_speed', 'torque', 'tool_wear', 'temp_diff', 'power_proxy', 'wear_per_speed', 'machine_type_H', 'machine_type_L', 'machine_type_M']


In [16]:
import pickle, os, config
with open('models/xgboost_final_model.pkl', 'rb') as f:
    model = pickle.load(f)
print(model.feature_names_in_)

['air_temp' 'proc_temp' 'rot_speed' 'torque' 'tool_wear' 'temp_diff'
 'power_proxy' 'wear_per_speed' 'machine_type_H' 'machine_type_L'
 'machine_type_M']


In [17]:
# Fixing the Logistic regression issue(multi class issue of not being fitted with it)

# Re-create the logistic regression model with the same parameters
lr = LogisticRegression(
    max_iter=1000,
    random_state=config.RANDOM_STATE,
    class_weight='balanced'
)

# Fit it on the balanced training data
lr.fit(X_train_bal, y_train_bal)

# Save it (overwrite the old file)
model_path = os.path.join(config.MODELS_DIR, "logistic_regression_final_model.pkl")
with open(model_path, 'wb') as f:
    pickle.dump(lr, f)

print("✅ LogisticRegression re-trained and saved.")

✅ LogisticRegression re-trained and saved.


In [18]:
# Verificaion if the fix was successfull

import pickle
with open('models/logistic_regression_final_model.pkl', 'rb') as f:
    lr = pickle.load(f)
print("Is fitted?", hasattr(lr, "coef_"))   # should be True
print("Has multi_class?", hasattr(lr, "multi_class"))  # should be True

Is fitted? True
Has multi_class? False


In [21]:
import pickle
import os
import config
from sklearn.linear_model import LogisticRegression

# 1. Create and fit the model
lr = LogisticRegression(
    max_iter=1000,
    random_state=config.RANDOM_STATE,
    class_weight='balanced'
)
lr.fit(X_train_bal, y_train_bal)   # make sure these variables exist

# 2. Save it (overwrites the old file)
model_path = os.path.join(config.MODELS_DIR, "logistic_regression_final_model.pkl")
with open(model_path, 'wb') as f:
    pickle.dump(lr, f)

print("✅ LogisticRegression re-fitted and saved.")

✅ LogisticRegression re-fitted and saved.


In [22]:
# Verificaion if the fix was successfull

import pickle
with open('models/logistic_regression_final_model.pkl', 'rb') as f:
    lr = pickle.load(f)
print("Is fitted?", hasattr(lr, "coef_"))   # should be True
print("Has multi_class?", hasattr(lr, "multi_class"))  # should be True

Is fitted? True
Has multi_class? False


In [24]:
import pickle
import os
import config
from sklearn.linear_model import LogisticRegression

# Resolving the multi-class issue
# 1. Create and fit the model (without multi_class in constructor)
lr = LogisticRegression(
    max_iter=1000,
    random_state=config.RANDOM_STATE,
    class_weight='balanced',
    solver='lbfgs'   # supports multinomial; works with auto
)
lr.fit(X_train_bal, y_train_bal)

# 2. Manually add the multi_class attribute (so inference code finds it)
lr.multi_class = 'auto'   # this will satisfy inference

# 3. Save it
model_path = os.path.join(config.MODELS_DIR, "logistic_regression_final_model.pkl")
with open(model_path, 'wb') as f:
    pickle.dump(lr, f)

print("✅ LogisticRegression re‑saved with multi_class attribute.")

✅ LogisticRegression re‑saved with multi_class attribute.


In [25]:
# Verificaion if the fix was successfull

import pickle
with open('models/logistic_regression_final_model.pkl', 'rb') as f:
    lr = pickle.load(f)
print("Is fitted?", hasattr(lr, "coef_"))   # should be True
print("Has multi_class?", hasattr(lr, "multi_class"))  # should be True

Is fitted? True
Has multi_class? True
